# "THE PRICE IS RIGHT" Capstone Project

This week - build a model that predicts how much something costs from a description, based on a scrape of Amazon data

# Order of play

DAY 1: Data Curation  
DAY 2: Data Pre-processing  
DAY 3: Evaluation, Baselines, Traditional ML  
DAY 4: Deep Learning and LLMs  
DAY 5: Fine-tuning a Frontier Model  

## DAY 5: Fine-tuning a Frontier Model

Now we will use OpenAI's API to fine-tune our own private variant of GPT-4.1-nano

In [1]:
# imports

import os
import re
import json
from dotenv import load_dotenv
from huggingface_hub import login
from openai import OpenAI
from pricer.items  import Item
from pricer.evaluator import evaluate

In [2]:
# environment

LITE_MODE = False

load_dotenv(override=True)
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [3]:
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

Loaded 800,000 training items, 10,000 validation items, 10,000 test items


In [4]:
openai = OpenAI()

# Data size

OpenAI recommends fine-tuning with a small population of 50-100 examples

I'm going to go with 20,000 points.

This cost me $3.42 - you should stick with 100 examples and the cost will be minimal!

In [5]:
# OpenAI recommends fine-tuning with populations of 50-100 examples
# But as our examples are very small, I'm suggesting we go with 100 examples (and 1 epoch)


fine_tune_train = train[:100]
fine_tune_validation = val[:50]

In [6]:
len(fine_tune_train)

100

# Step 1

Prepare our data for fine-tuning in JSONL (JSON Lines) format and upload to OpenAI

In [8]:
def messages_for(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [
        {"role": "user", "content": message},
        {"role": "assistant", "content": f"${item.price:.2f}"}
    ]

In [9]:
messages_for(fine_tune_train[0])

[{'role': 'user',
  'content': 'Estimate the price of this product. Respond with the price, no explanation\n\nTitle: Schlage F59 & 613 Andover Interior Knob (Deadbolt Included)  \nCategory: Home Hardware  \nBrand: Schlage  \nDescription: A single‑piece oil‑rubbed bronze knob that mounts to a deadbolt for secure, easy interior door use.  \nDetails: Designed for a 4" minimum center‑to‑center door prep, it offers a lifetime mechanical and finish warranty and comes ready for quick installation.'},
 {'role': 'assistant', 'content': '$64.30'}]

In [10]:
# Convert the items into a list of json objects - a "jsonl" string
# Each row represents a message in the form:
# {"messages" : [{"role": "system", "content": "You estimate prices...


def make_jsonl(items):
    result = ""
    for item in items:
        messages = messages_for(item)
        messages_str = json.dumps(messages)
        result += '{"messages": ' + messages_str +'}\n'
    return result.strip()

In [11]:
print(make_jsonl(train[:3]))

{"messages": [{"role": "user", "content": "Estimate the price of this product. Respond with the price, no explanation\n\nTitle: Schlage F59 & 613 Andover Interior Knob (Deadbolt Included)  \nCategory: Home Hardware  \nBrand: Schlage  \nDescription: A single\u2011piece oil\u2011rubbed bronze knob that mounts to a deadbolt for secure, easy interior door use.  \nDetails: Designed for a 4\" minimum center\u2011to\u2011center door prep, it offers a lifetime mechanical and finish warranty and comes ready for quick installation."}, {"role": "assistant", "content": "$64.30"}]}
{"messages": [{"role": "user", "content": "Estimate the price of this product. Respond with the price, no explanation\n\nTitle: Mini Electric Air Duster Fan  \nCategory: Electronics  \nBrand: Kica  \nDescription: Ultra\u2011compact 86,000\u202fRPM electric air duster with 11\u202fm/s wind speed for precise cleaning and inflation.  \nDetails: Powered by a 9.99\u202fWh motor, adjustable in four speed levels, it uses three 

In [12]:
# Convert the items into jsonl and write them to a file

def write_jsonl(items, filename):
    with open(filename, "w") as f:
        jsonl = make_jsonl(items)
        f.write(jsonl)

In [13]:
write_jsonl(fine_tune_train, "jsonl/fine_tune_train.jsonl")

In [14]:
write_jsonl(fine_tune_validation, "jsonl/fine_tune_validation.jsonl")

In [15]:
with open("jsonl/fine_tune_train.jsonl", "rb") as f:
    train_file = openai.files.create(file=f, purpose="fine-tune")

In [27]:
train_file

FileObject(id='file-4PCRzDU8MTt5CoNwesjUFh', bytes=55219, created_at=1771970768, filename='fine_tune_train.jsonl', object='file', purpose='fine-tune', status='processed', expires_at=None, status_details=None)

In [28]:
with open("jsonl/fine_tune_validation.jsonl", "rb") as f:
    validation_file = openai.files.create(file=f, purpose="fine-tune")

In [32]:
validation_file
file_status = openai.files.retrieve(train_file.id)
print(file_status.status)

processed


https://platform.openai.com/storage/files/

# Step 2

## And now time to Fine-tune!

In [33]:
openai.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=validation_file.id,
    model="gpt-4.1-nano-2025-04-14",
    seed=42,
    hyperparameters={"n_epochs": 1, "batch_size": 1},
    suffix="pricer"
)

FineTuningJob(id='ftjob-5SmpxLVmvxPtzLFkifIMLNbt', created_at=1771971512, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(batch_size=1, learning_rate_multiplier='auto', n_epochs=1), model='gpt-4.1-nano-2025-04-14', object='fine_tuning.job', organization_id='org-hDUaiYrQsnG45Qt4OOJYarc3', result_files=[], seed=42, status='validating_files', trained_tokens=None, training_file='file-4PCRzDU8MTt5CoNwesjUFh', validation_file='file-MQdAAaEs4DbC85pztZHMDL', estimated_finish=None, integrations=[], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size=1, learning_rate_multiplier='auto', n_epochs=1))), user_provided_suffix='pricer', usage_metrics=None, shared_with_openai=False, eval_id=None, internal_worker_backend=None)

In [ ]:
job_id = "ftjob-5SmpxLVmvxPtzLFkifIMLNbt"
job = openai.fine_tuning.jobs.retrieve(job_id)
print("status:", job.status)

status: queued


In [46]:
openai.fine_tuning.jobs.list(limit=20)

SyncCursorPage[FineTuningJob](data=[FineTuningJob(id='ftjob-5SmpxLVmvxPtzLFkifIMLNbt', created_at=1771971512, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(batch_size=1, learning_rate_multiplier=0.1, n_epochs=1), model='gpt-4.1-nano-2025-04-14', object='fine_tuning.job', organization_id='org-hDUaiYrQsnG45Qt4OOJYarc3', result_files=[], seed=42, status='queued', trained_tokens=None, training_file='file-4PCRzDU8MTt5CoNwesjUFh', validation_file='file-MQdAAaEs4DbC85pztZHMDL', estimated_finish=None, integrations=[], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size=1, learning_rate_multiplier=0.1, n_epochs=1))), user_provided_suffix='pricer', usage_metrics=None, shared_with_openai=False, eval_id=None, internal_worker_backend=None), FineTuningJob(id='ftjob-X4Vg9qieUdzUUJl0uc0wKPhJ', created_at=1771970890, erro

In [47]:
job_id = openai.fine_tuning.jobs.list(limit=1).data[0].id

In [48]:
job_id

'ftjob-5SmpxLVmvxPtzLFkifIMLNbt'

In [63]:
job=openai.fine_tuning.jobs.retrieve(job_id)
print("status:", job.status)

status: succeeded


In [64]:
openai.fine_tuning.jobs.list_events(fine_tuning_job_id=job_id, limit=100).data

[FineTuningJobEvent(id='ftevent-CoOvNzS4zEXIZIix7xmFxfZk', created_at=1771973274, level='info', message='The job has successfully completed', object='fine_tuning.job.event', data={}, type='message'),
 FineTuningJobEvent(id='ftevent-LL476njdTVIThMmvQCYvcJOp', created_at=1771973273, level='info', message='Usage policy evaluations completed, model is now enabled for sampling', object='fine_tuning.job.event', data={}, type='message'),
 FineTuningJobEvent(id='ftevent-SAK0Jc0BWr0GxM5H7UHqSmFm', created_at=1771973273, level='info', message='Moderation checks for snapshot ft:gpt-4.1-nano-2025-04-14:personal:pricer:DCvIJms1 passed.', object='fine_tuning.job.event', data={'blocked': False, 'results': [{'flagged': False, 'category': 'harassment/threatening', 'enforcement': 'blocking'}, {'flagged': False, 'category': 'sexual', 'enforcement': 'blocking'}, {'flagged': False, 'category': 'sexual/minors', 'enforcement': 'blocking'}, {'flagged': False, 'category': 'propaganda', 'enforcement': 'blocking

https://platform.openai.com/finetune


# Step 3

Test our fine tuned model

In [65]:
fine_tuned_model_name = openai.fine_tuning.jobs.retrieve(job_id).fine_tuned_model 
    

In [66]:
fine_tuned_model_name

'ft:gpt-4.1-nano-2025-04-14:personal:pricer:DCvIJms1'

In [67]:
# The prompt

def test_messages_for(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [
        {"role": "user", "content": message},
    ]

In [68]:
# Try this out

test_messages_for(test[0])

[{'role': 'user',
  'content': 'Estimate the price of this product. Respond with the price, no explanation\n\nTitle: Excess V2 Distortion/Modulation Pedal  \nCategory: Music Pedals  \nBrand: Old Blood Noise  \nDescription: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  \nDetails: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switching, soft‑touch bypass, and expression jack for dynamic control.'}]

In [69]:
# The inference function


def gpt_4__1_nano_fine_tuned(item):
    response = openai.chat.completions.create(
        model=fine_tuned_model_name,
        messages=test_messages_for(item),
        max_tokens=7
    )
    return response.choices[0].message.content

In [70]:
print(test[0].price)
print(gpt_4__1_nano_fine_tuned(test[0]))

219.0
$149.00


In [71]:
evaluate(gpt_4__1_nano_fine_tuned, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$20 $217 $15 $26 $123 $43 $2 $54 $26 $37 $97 $320 $15 $7 $9 $7 $41 $0 $160 $75 $3 $76 $112 $1 $132 $274 $215 $2 $105 $60 $22 $2 $87 $13 $37 $240 $50 $24 $34 $4 $162 $10 $57 $142 $21 $0 $23 $8 $59 $77 $22 $112 $121 $10 $267 $40 $13 $22 $76 $17 $133 $28 $235 $30 $47 $30 $141 $311 $30 $50 $18 $20 $210 $13 $37 $26 $37 $8 $6 $5 $71 $12 $50 $74 $2 $15 $21 $86 $10 $72 $15 $3 $34 $17 $3 $71 $10 $424 $72 $14 $42 $228 $7 $66 $2914 $2 $17 $324 $6 $70 $30 $308 $144 $29 $4 $31 $25 $17 $14 $20 $29 $408 $74 $105 $14 $11 $36 $67 $13 $112 $318 $156 $109 $4 $145 $8 $7 $30 $14 $37 $68 $89 $38 $5 $104 $205 $50 $11 $135 $18 $2 $164 $11 $73 $16 $134 $17 $12 $19 $16 $52 $16 $45 $2 $246 $51 $205 $39 $21 $2 $9 $6 $13 $7 $36 $171 $10 $23 $44 $16 $293 $5 $240 $126 $49 $7 $22 $36 $10 $3 $10 $84 $97 $61 $60 $210 $59 $106 $14 $12 

In [ ]:
# 96.58 - mini 200
# 79.29 - mini 2000
# 82.26 - nano 2000
# 67.75 - nano 20,000